# Homogeneous elastic 2-D convergence viewer

This lightweight notebook reads the compact 500/375/250 m products generated by `scripts/run_flat_free_surface_convergence.jl`. It does not rerun FD, OPT or SPECFEM2D.

In [ ]:
import Pkg
function find_flexopt_root(start=pwd())
    directory = abspath(start)
    while true
        isfile(joinpath(directory, "src", "flexOPT.jl")) && return directory
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    haskey(ENV, "FLEXOPT_ROOT") && return abspath(ENV["FLEXOPT_ROOT"])
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .specfemBenchmark
using CairoMakie, Statistics, JLD2, JSON
CairoMakie.activate!(type="png")
@show VERSION Threads.nthreads() Base.active_project()


## Self-convergence and matched-resolution OPT/SPECFEM comparison

In [ ]:
sourceNotebook = JSON.parsefile(joinpath(flexopt_root, "notebooks",
    "HomogeneousElastic2DBenchmark_FreeSurface.ipynb"))
viewerCell = only(filter(cell -> get(cell, "id", "") ==
    "cross-convergence-viewer", sourceNotebook["cells"]))
Base.include_string(Main, join(viewerCell["source"]),
    "HomogeneousElastic2DBenchmark_FreeSurface:convergence-viewer")


## Station-by-station convergence and semi-analytical arrivals

This diagnostic keeps every receiver visible. Solid arrival markers are direct P/S arrivals; dashed markers are the exact homogeneous PP, PS, SP and SS stationary-ray times obtained from Snell's law.

In [ ]:
include(joinpath(flexopt_root, "scripts",
    "plot_flat_free_surface_diagnostics.jl"))
freeSurfaceDiagnostics = main()
display(freeSurfaceDiagnostics.specfem_station_convergence)
display(freeSurfaceDiagnostics.specfem_ux_normalized)
display(freeSurfaceDiagnostics.specfem_uz_normalized)
display(freeSurfaceDiagnostics.finest_cross_method)
display(freeSurfaceDiagnostics.fullspace_green_gate)
display(freeSurfaceDiagnostics.all_methods_ux_absolute)
display(freeSurfaceDiagnostics.all_methods_uz_absolute)
display(freeSurfaceDiagnostics.late_ux_absolute)
display(freeSurfaceDiagnostics.late_uz_absolute)
display(freeSurfaceDiagnostics.coupled_space_time_convergence)


## Reading the result

The vertical component converges strongly between OPT3 and SPECFEM2D. For `uₓ`, do not average relative errors at symmetry stations: the physical reference is zero there. The pre-surface Green2D gate shows good agreement away from the source-aligned receiver column, while the oscillatory SPECFEM `uₓ` on that column points to a receiver/component or mesh-symmetry issue that must be audited before treating SPECFEM as the absolute horizontal reference.